# Modelo LNR v2 — Detector de Bulos sobre Inmigración

Versión mejorada respecto a `entrenar_modelo.ipynb`:

| Mejora | Descripción |
|--------|-------------|
| **Modelo** | `microsoft/mdeberta-v3-base` en lugar de `xlm-roberta-base` (+3-5 F1) |
| **Cross-encoder** | Titular y texto como dos secuencias separadas por `[SEP]`, replicando el formato de inferencia RAG |
| **ALERTA corregida** | `ALERTA → FALSO` (coherente con el pipeline de inferencia) |
| **Oversampling CONTEXTO** | CONTEXTO repetido ×4 en train para compensar la escasez de ejemplos |
| **Focal Loss** | Foca el gradiente en ejemplos difíciles; reduce el sesgo hacia VERDADERO |
| **Pesos manuales** | CONTEXTO ×8, FALSO ×3 (más agresivos que los proporcionales automáticos) |

**Datasets necesarios** (mismos que v1):
- `dataset_inmigracion_unificado_limpio_tema.xlsx`
- `dataset_inmigracion_falso_contexto.xlsx`

**Requisitos:** GPU T4 o superior (Google Colab / Kaggle)

---
### Como subir los datos
**Colab:** Sube los dos `.xlsx` a `Mi unidad/LNR_datos/` en Google Drive.  
**Kaggle:** Crea un dataset llamado `lnr-inmigracion` con los dos `.xlsx` y adjuntalo al notebook.

In [ ]:
# 1. INSTALACION DE DEPENDENCIAS
!pip install -q "transformers[torch]>=4.40" accelerate scikit-learn openpyxl

In [ ]:
# 2. CONFIGURACION
import os

os.environ['PYTORCH_ALLOC_CONF']    = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Detectar entorno
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
IS_KAGGLE = os.path.exists('/kaggle/input')

# Rutas
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR   = '/content/drive/MyDrive/LNR_datos'
    OUTPUT_DIR = '/content/drive/MyDrive/LNR_modelo_v2'
elif IS_KAGGLE:
    DATA_DIR   = '/kaggle/input/lnr-inmigracion'
    OUTPUT_DIR = '/kaggle/working/modelo_lnr_v2'
else:
    DATA_DIR   = './DATOS'
    OUTPUT_DIR = './modelo_lnr_v2'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Modelo
# mdeberta-v3-base supera a xlm-roberta-base en clasificacion multilingue.
# Alternativa con mas VRAM (>=16 GB): 'xlm-roberta-large'
MODEL_NAME = 'microsoft/mdeberta-v3-base'

# Hiperparametros
MAX_LEN                 = 512
BATCH_SIZE              = 16   # reducido respecto a v1; mdeberta usa mas VRAM
GRAD_ACCUMULATION_STEPS = 2    # batch efectivo = 16 x 2 = 32
EPOCHS                  = 8
LR                      = 1e-5
WARMUP_RATIO            = 0.1  # se convierte a warmup_steps antes de entrenar
WEIGHT_DECAY            = 0.01
LABEL_SMOOTHING         = 0.1  # regularizacion: evita sobreconfianza

# Pesos de clase: calculados automaticamente en celda 9 a partir de la
# distribucion real del train set. No se usan pesos manuales: la combinacion
# de oversampling extremo + pesos altos provocó colapso hacia CONTEXTO en
# pruebas previas (el modelo predecia CONTEXTO para todo).

LABEL2ID = {'VERDADERO': 0, 'CONTEXTO': 1, 'FALSO': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

import torch
n_gpus = torch.cuda.device_count()
print(f'Entorno:          {"Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else "Local"}')
print(f'GPUs disponibles: {n_gpus}')
print(f'Modelo:           {MODEL_NAME}')
print(f'Batch efectivo:   {BATCH_SIZE * max(n_gpus,1) * GRAD_ACCUMULATION_STEPS}')
print(f'Datos:            {DATA_DIR}')
print(f'Salida:           {OUTPUT_DIR}')

In [ ]:
# 3. IMPORTS Y VERIFICACION DE GPU
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'PyTorch:  {torch.__version__}')
print(f'CUDA:     {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU:      {gpu.name}')
    print(f'VRAM:     {gpu.total_memory / 1e9:.1f} GB')
else:
    print('Sin GPU — activa la T4 en Runtime > Change runtime type')

In [ ]:
# 4. CARGA Y EXPLORACION DE DATOS
FILE_REAL = os.path.join(DATA_DIR, 'dataset_inmigracion_unificado_limpio_tema.xlsx')
FILE_FAKE = os.path.join(DATA_DIR, 'dataset_inmigracion_falso_contexto.xlsx')

df_real = pd.read_excel(FILE_REAL)
df_fake = pd.read_excel(FILE_FAKE)

print('── Dataset real (noticias scrapeadas) ──')
print(df_real['etiqueta'].value_counts().to_string())
print(f'   Total: {len(df_real)}')

print('\n── Dataset IA-generado (adulteraciones) ──')
print(df_fake['etiqueta'].value_counts().to_string())
print(f'   Total: {len(df_fake)}')

print('\n── Tecnicas de manipulacion usadas ──')
print(df_fake['tecnica'].value_counts().head(8).to_string())

len_total = (df_real['titulo'].str.len() + df_real['texto'].str.len())
print(f'\n── Longitud de textos (chars, dataset real) ──')
print(f'   media:   {len_total.mean():.0f}')
print(f'   mediana: {len_total.median():.0f}')
print(f'   p90:     {len_total.quantile(0.9):.0f}')

In [ ]:
# 5. PREPARACION DEL DATASET COMBINADO
#
# CAMBIO v2: ALERTA -> FALSO (no VERDADERO como en v1).
# Razon: en el pipeline de inferencia, 'alerta' es tratado como tipo fact-checker
# (bulo detectado por Maldita). Mapear a VERDADERO en train era inconsistente
# con como el clasificador los usa en produccion.

# Dataset 1: noticias reales
df1 = df_real[['titulo', 'texto', 'etiqueta']].copy()
df1['etiqueta'] = df1['etiqueta'].replace('ALERTA', 'FALSO')   # corregido respecto a v1
df1 = df1[df1['etiqueta'].isin(LABEL2ID)]

# Dataset 2: versiones adulteradas por IA (FALSO / CONTEXTO)
df2 = df_fake[['titulo', 'texto', 'etiqueta']].copy()
df2 = df2[df2['etiqueta'].isin(LABEL2ID)]

# Combinar y limpiar
df = pd.concat([df1, df2], ignore_index=True)
df = df.dropna(subset=['titulo', 'texto'])
df['titulo'] = df['titulo'].astype(str).str.strip()
df['texto']  = df['texto'].astype(str).str.strip()
df = df[(df['titulo'].str.len() > 5) & (df['texto'].str.len() > 20)]
df['label'] = df['etiqueta'].map(LABEL2ID).astype(int)
df = df.reset_index(drop=True)

print('── Distribucion final ──')
for etiq, cnt in df['etiqueta'].value_counts().items():
    bar = '#' * (cnt // 100)
    print(f'  {etiq:<12} {cnt:5d}  {cnt/len(df)*100:5.1f}%  {bar}')
print(f'\n  TOTAL        {len(df):5d}')

In [ ]:
# 6. SPLIT TRAIN / VAL / TEST (80% / 10% / 10%)
train_df, test_df = train_test_split(
    df, test_size=0.10, random_state=SEED, stratify=df['label']
)
train_df, val_df = train_test_split(
    train_df, test_size=0.111, random_state=SEED, stratify=train_df['label']
)  # 0.111 aprox 10% del total

print(f'{"Split":<8} {"N":>6}  {"VERDADERO":>10}  {"CONTEXTO":>9}  {"FALSO":>7}')
print('-' * 50)
for nombre, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    c = split['etiqueta'].value_counts()
    print(f'{nombre:<8} {len(split):>6}  '
          f'{c.get("VERDADERO", 0):>10}  '
          f'{c.get("CONTEXTO", 0):>9}  '
          f'{c.get("FALSO", 0):>7}')

In [ ]:
# 7. TOKENIZADOR Y CLASE DATASET (cross-encoder)
#
# Formato cross-encoder: tokenizer(titulo, texto) — dos secuencias separadas por [SEP].
# Esto replica el formato de inferencia RAG: afirmacion [SEP] documento_recuperado.
# truncation='only_second' garantiza que el titular (afirmacion) nunca se trunca.
#
# NO se aplica oversampling de CONTEXTO: la combinacion oversampling x4 + peso x8
# produjo colapso hacia CONTEXTO (gradiente efectivo ~32x). Los pesos proporcionales
# calculados en la celda siguiente equilibran las clases sin necesidad de duplicar.

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class NoticiaDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.titulos = dataframe['titulo'].values.tolist()
        self.textos  = dataframe['texto'].values.tolist()
        self.labels  = dataframe['label'].values.tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.titulos[idx],        # secuencia A: afirmacion / titular
            self.textos[idx],         # secuencia B: evidencia / cuerpo
            max_length=self.max_len,
            truncation='only_second', # nunca truncar la afirmacion
            padding='max_length',
            return_tensors='pt',
        )
        item = {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }
        if 'token_type_ids' in enc:
            item['token_type_ids'] = enc['token_type_ids'].squeeze()
        return item


train_ds = NoticiaDataset(train_df, tokenizer, MAX_LEN)
val_ds   = NoticiaDataset(val_df,   tokenizer, MAX_LEN)
test_ds  = NoticiaDataset(test_df,  tokenizer, MAX_LEN)

s = train_ds[0]
print(f'input_ids shape: {s["input_ids"].shape}')
print(f'label:           {ID2LABEL[s["labels"].item()]}')
print(f'titular ejemplo: {train_df["titulo"].iloc[0][:70]}')

In [ ]:
# 8. MODELO
import gc
if 'model' in dir():
    del model
torch.cuda.empty_cache()
gc.collect()

if torch.cuda.is_available():
    vram_libre = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
    print(f'VRAM libre antes de cargar: {vram_libre / 1e9:.2f} GB')

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

n_params = sum(p.numel() for p in model.parameters())
print(f'Modelo:              {MODEL_NAME}')
print(f'Parametros totales:  {n_params:,}')

if torch.cuda.is_available():
    vram_modelo = torch.cuda.memory_allocated(0)
    print(f'VRAM tras cargar:    {vram_modelo / 1e9:.2f} GB')

In [ ]:
# 9. METRICAS, PESOS Y TRAINER
#
# Pesos proporcionales calculados a partir de la distribucion real del train set.
# Formula: len(train) / (n_clases * count_clase) -> todas las clases contribuyen
# igual al loss total, sin importar su frecuencia.
# Esto es mas estable que pesos manuales extremos (que provocaron colapso).

counts  = train_df['label'].value_counts().sort_index().values
weights = torch.tensor(
    len(train_df) / (len(LABEL2ID) * counts), dtype=torch.float
)
print('Pesos de clase (proporcionales automaticos):')
for i, w in enumerate(weights):
    print(f'  {ID2LABEL[i]:<12} {w:.4f}  (N={counts[i]})')
print(f'\nLabel smoothing: {LABEL_SMOOTHING}')


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    from collections import Counter
    dist = Counter(preds.tolist())
    dist_str = '  '.join(f'{ID2LABEL[k]}:{v}' for k, v in sorted(dist.items()))
    print(f'  Distribucion predicciones: {dist_str}')
    return {
        'accuracy':    float(accuracy_score(labels, preds)),
        'f1_macro':    float(f1_score(labels, preds, average='macro')),
        'f1_weighted': float(f1_score(labels, preds, average='weighted')),
        'f1_contexto': float(f1_score(labels, preds, labels=[1], average='macro')),
        'f1_falso':    float(f1_score(labels, preds, labels=[2], average='macro')),
    }


class WeightedTrainer(Trainer):
    """
    CrossEntropy ponderada + label smoothing.

    Se usa CrossEntropy (no Focal Loss) porque en pruebas previas
    Focal Loss con gamma=2 + pesos extremos provoco colapso de clase.
    CrossEntropy + label smoothing es mas estable con datasets pequeños.
    """
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        # Nota: weights debe tener el mismo dtype que los logits.
        # Con BF16 activo, logits son bfloat16; con FP32, son float32.
        # .to(device=..., dtype=...) garantiza compatibilidad en ambos casos.
        w = weights.to(device=outputs.logits.device, dtype=outputs.logits.dtype)
        loss_fn = torch.nn.CrossEntropyLoss(
            weight=w,
            label_smoothing=LABEL_SMOOTHING,
        )
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


# Callback para corregir el bug de mdeberta-v3-base con claves LayerNorm.
# El checkpoint se guarda con 'gamma'/'beta' (nombres antiguos) pero el modelo
# espera 'weight'/'bias'. Sin este fix, load_best_model_at_end carga LayerNorm
# con pesos aleatorios (las claves no coinciden y se silencia el error).
from transformers import TrainerCallback

class FixLayerNormCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        ckpt_dir = os.path.join(args.output_dir, f'checkpoint-{state.global_step}')
        sf_path  = os.path.join(ckpt_dir, 'model.safetensors')
        pt_path  = os.path.join(ckpt_dir, 'pytorch_model.bin')

        if os.path.exists(sf_path):
            try:
                from safetensors.torch import load_file, save_file
                sd = load_file(sf_path)
                new_sd, renamed = {}, 0
                for k, v in sd.items():
                    nk = k.replace('LayerNorm.gamma', 'LayerNorm.weight') \
                          .replace('LayerNorm.beta',  'LayerNorm.bias')
                    if nk != k:
                        renamed += 1
                    new_sd[nk] = v
                if renamed > 0:
                    save_file(new_sd, sf_path)
                    print(f'  [FixLayerNorm] {renamed} claves renombradas en {ckpt_dir}')
            except Exception as e:
                print(f'  [FixLayerNorm] Error con safetensors: {e}')

        elif os.path.exists(pt_path):
            sd = torch.load(pt_path, map_location='cpu')
            new_sd, renamed = {}, 0
            for k, v in sd.items():
                nk = k.replace('LayerNorm.gamma', 'LayerNorm.weight') \
                      .replace('LayerNorm.beta',  'LayerNorm.bias')
                if nk != k:
                    renamed += 1
                new_sd[nk] = v
            if renamed > 0:
                torch.save(new_sd, pt_path)
                print(f'  [FixLayerNorm] {renamed} claves renombradas en {ckpt_dir}')

In [ ]:
# 10. ENTRENAMIENTO

# Calcular warmup_steps a partir de warmup_ratio para evitar el aviso de deprecacion
n_gpus         = torch.cuda.device_count()
steps_per_epoch = len(train_ds) // (BATCH_SIZE * max(n_gpus, 1))
total_steps     = steps_per_epoch * EPOCHS // GRAD_ACCUMULATION_STEPS
warmup_steps    = int(total_steps * WARMUP_RATIO)
print(f'Total steps:   {total_steps}')
print(f'Warmup steps:  {warmup_steps}  ({WARMUP_RATIO:.0%})')

training_args = TrainingArguments(
    output_dir                      = OUTPUT_DIR,
    num_train_epochs                = EPOCHS,
    per_device_train_batch_size     = BATCH_SIZE,
    per_device_eval_batch_size      = 32,
    gradient_accumulation_steps     = GRAD_ACCUMULATION_STEPS,
    warmup_steps                    = warmup_steps,
    weight_decay                    = WEIGHT_DECAY,
    learning_rate                   = LR,
    lr_scheduler_type               = 'cosine',
    eval_strategy                   = 'epoch',
    save_strategy                   = 'epoch',
    save_total_limit                = 1,
    load_best_model_at_end          = True,
    metric_for_best_model           = 'f1_macro',
    greater_is_better               = True,
    # mdeberta-v3 usa atencion FP16 internamente: fp16=True choca con GradScaler.
    # bf16 funciona en A100/H100; en T4 entrena en FP32 (mas lento, pero correcto).
    fp16                            = False,
    bf16                            = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    # use_reentrant=False necesario para gradient_checkpointing con mdeberta
    gradient_checkpointing          = True,
    gradient_checkpointing_kwargs   = {'use_reentrant': False},
    dataloader_pin_memory           = True,
    dataloader_num_workers          = 2,
    logging_steps                   = 50,
    report_to                       = 'none',
    seed                            = SEED,
)

trainer = WeightedTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [
        EarlyStoppingCallback(early_stopping_patience=3),
        FixLayerNormCallback(),   # renombra gamma/beta -> weight/bias en checkpoints
    ],
)

batch_efectivo = BATCH_SIZE * max(n_gpus, 1) * GRAD_ACCUMULATION_STEPS
usa_bf16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
print(f'\nModelo:            {MODEL_NAME}')
print(f'GPUs activos:      {n_gpus}')
print(f'Batch por GPU:     {BATCH_SIZE}')
print(f'Batch efectivo:    {batch_efectivo}')
print(f'Precision:         {"BF16" if usa_bf16 else "FP32 (T4 no soporta BF16)"}')
print(f'Steps por epoca:   {steps_per_epoch}')
print(f'Epocas max:        {EPOCHS}  (early stop si no mejora 3 epocas en f1_macro)')
print(f'Train samples:     {len(train_ds)}')
print()
trainer.train()

In [ ]:
# 11. EVALUACION EN TEST SET
output = trainer.predict(test_ds)
y_pred = np.argmax(output.predictions, axis=-1)
y_true = output.label_ids

target_names = [ID2LABEL[i] for i in sorted(ID2LABEL)]

print('=' * 60)
print('RESULTADOS EN CONJUNTO DE TEST')
print('=' * 60)
print(classification_report(
    y_true, y_pred,
    target_names=target_names,
    digits=3
))

cm = confusion_matrix(y_true, y_pred)
print('Matriz de confusion (filas = real, columnas = predicho):')
header = f"{'':15}" + ''.join(f'{n:>12}' for n in target_names)
print(header)
for i, row in enumerate(cm):
    print(f'  {target_names[i]:<13}' + ''.join(f'{v:>12}' for v in row))

In [ ]:
# 12. GUARDAR MODELO
MODEL_SAVE = os.path.join(OUTPUT_DIR, 'modelo_final')
os.makedirs(MODEL_SAVE, exist_ok=True)

trainer.save_model(MODEL_SAVE)
tokenizer.save_pretrained(MODEL_SAVE)

print(f'Modelo guardado en: {MODEL_SAVE}')
print('\nArchivos generados:')
for f in sorted(os.listdir(MODEL_SAVE)):
    size = os.path.getsize(os.path.join(MODEL_SAVE, f))
    size_str = f'{size/1e6:.1f} MB' if size > 1e5 else f'{size/1e3:.0f} KB'
    print(f'  {f:<40} {size_str}')

# En Colab: los archivos quedan en Drive (OUTPUT_DIR/modelo_final/).
# En Kaggle: descargar desde Output > modelo_lnr_v2/modelo_final/

In [ ]:
# 13. TEST DE INFERENCIA (cross-encoder)
#
# En produccion el clasificador recibe dos cadenas:
#   - texto_a: la afirmacion del usuario (query)
#   - texto_b: el documento recuperado del RAG
# Este test replica ese formato.

from transformers import pipeline

pipe = pipeline(
    'text-classification',
    model=MODEL_SAVE,
    tokenizer=MODEL_SAVE,
    device=0 if torch.cuda.is_available() else -1,
    truncation=True,
    max_length=MAX_LEN,
)

EMOJIS = {'VERDADERO': '[V]', 'CONTEXTO': '[C]', 'FALSO': '[F]'}

# Pares (afirmacion, fragmento_de_evidencia)
# En produccion el segundo elemento viene del RAG; aqui usamos textos de prueba.
ejemplos = [
    (
        'Los inmigrantes ilegales cobran 4.700 euros al mes',
        'Maldita Migracion desmiente que los menores migrantes reciban 4700 euros. '
        'La cifra real es la plaza de acogida en un centro, no una transferencia economica.'
    ),
    (
        'El Gobierno aprueba medidas para agilizar los expedientes de asilo',
        'El Consejo de Ministros ha aprobado un paquete de medidas para reducir la '
        'burocracia en los tramites de asilo y acelerar las resoluciones pendientes.'
    ),
    (
        'Las llegadas de migrantes a Canarias se han disparado este anyo',
        'El Ministerio del Interior registro un aumento del 40% en llegadas irregulares '
        'a Canarias respecto al mismo periodo del anyo anterior, segun datos oficiales.'
    ),
    (
        'Los menas cometen el 80% de los delitos en las ciudades donde viven',
        'Maldita Migracion: falso. Segun el Ministerio del Interior, los menores '
        'extranjeros no acompakyados representan menos del 1% de los detenidos en Espanya.'
    ),
]

print(f'{"Veredicto":<18} {"Conf":>6}  Afirmacion')
print('-' * 75)
for afirmacion, evidencia in ejemplos:
    # El pipeline recibe la concatenacion; en ClasificadorLNR se separan
    # internamente via tokenizer(text_a, text_b)
    entrada = afirmacion + ' ' + evidencia
    res     = pipe(entrada)[0]
    emoji   = EMOJIS.get(res['label'], '[ ]')
    print(f'{emoji} {res["label"]:<14} {res["score"]:>5.1%}  {afirmacion[:55]}')

## Pasos siguientes

### 1. Copiar el modelo entrenado al proyecto

Descarga la carpeta `modelo_final/` desde Drive o Kaggle Output y sustituye `PROY III/modelo_lnr/modelo_final/`.

### 2. Actualizar `clasificador_lnr.py`

Con el nuevo modelo entrenado en formato cross-encoder, el postprocesado agresivo de v1 (Focal boost, VERDADERO override, etc.) puede suavizarse porque el modelo ya distingue mejor por si solo. Recomendacion de parametros de partida:

```python
UMBRAL_CONFIANZA_DEFAULT   = 0.50
UMBRAL_FACTCHECK_TOP       = 0.45
PESO_FACTCHECK             = 1.40   # menos agresivo; el modelo ya aprende el contraste
SEÑAL_BULO_BOOST           = 1.40
PENALIZACION_VERDADERO_TOP = 0.88
```

### 3. Volver a evaluar

```bash
python evaluar_chatbot.py --api-key TU_KEY --solo-curado --verbose
```

### 4. Si OOM en la T4 con mdeberta-v3-base

Reduce en la celda 2:
```python
BATCH_SIZE              = 8
GRAD_ACCUMULATION_STEPS = 4   # batch efectivo sigue siendo 32
```